# Phase 6 — FastAPI query service + Flask ingestion service

Exercises both services' routes with each framework's own test client,
sharing one `RetrievalPipeline` instance so this notebook process only
opens the local-mode Qdrant storage once. See
`learning/08_api_services/01_fastapi_flask_services.md` for why these are
tested this way rather than as two live concurrent servers.

## Shared pipeline (empty collection to start)

Starts from an empty collection on purpose, so `/ingest` below is what
actually populates it — proving the ingestion service, not notebook setup
code, does the real work this time.

In [1]:
import logging
import os
import sys
from pathlib import Path

logging.getLogger("numexpr").setLevel(logging.WARNING)
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("huggingface_hub").setLevel(logging.WARNING)
logging.getLogger("openai").setLevel(logging.WARNING)

sys.path.insert(0, str(Path.cwd().parent))

from src.retrieval import RetrievalPipeline

STORAGE_PATH = Path("data/qdrant_api_demo")
pipeline = RetrievalPipeline(storage_path=STORAGE_PATH)
print("Pipeline ready (collection not created yet — /ingest will create it)")

/Users/rtlecl910010/Desktop/Projects/Project/papermind/backend/.venv/lib/python3.11/site-packages/huggingface_hub/utils/tqdm.py:176: UserWarning: Cannot enable progress bars: environment variable `HF_HUB_DISABLE_PROGRESS_BARS=1` is set and has priority.
  warnings.warn(


Pipeline ready (collection not created yet — /ingest will create it)


## Flask ingestion service — `/health` and `/ingest`

In [2]:
from generate_sample_pdf import generate

from src.services import create_ingestion_app

ingestion_app = create_ingestion_app(pipeline)
ingestion_client = ingestion_app.test_client()

print("health:", ingestion_client.get("/health").get_json())

pdf_path = generate()
with open(pdf_path, "rb") as pdf_file:
    response = ingestion_client.post(
        "/ingest",
        data={"file": (pdf_file, "sample.pdf")},
        content_type="multipart/form-data",
    )
print("ingest status:", response.status_code)
print("ingest body:", response.get_json())

health: {'status': 'ok'}


ingest status: 200
ingest body: {'chunks': 4}


## FastAPI query service — `/health` and `/ask`

Needs `backend/.env` with `OPENAI_API_KEY`. Uses the same `pipeline`
instance (now populated by `/ingest` above), so no second Qdrant client
opens the same storage path.

In [3]:
from fastapi.testclient import TestClient

from src.services import create_query_app

query_app = create_query_app(pipeline)
query_client = TestClient(query_app)

print("health:", query_client.get("/health").json())

ask_response = query_client.post(
    "/ask", json={"question": "How is citation accuracy measured in this paper?"}
)
print("ask status:", ask_response.status_code)
print("ask body:", ask_response.json())

2026-11-25 19:26:55,284 - INFO - HTTP Request: GET http://testserver/health "HTTP/1.1 200 OK"


health: {'status': 'ok'}


2026-11-25 19:26:57,198 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


2026-11-25 19:26:58,752 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


2026-11-25 19:26:58,777 - INFO - HTTP Request: POST http://testserver/ask "HTTP/1.1 200 OK"


ask status: 200
ask body: {'answer': 'The context does not provide specific information on how citation accuracy is measured in the paper. It only mentions that dense retrieval combined with a re-ranking stage improves citation accuracy compared to retrieval alone (page 1) and that grounded answers with page citations were rated as more trustworthy than ungrounded answers by human reviewers (page 4). However, it does not detail the methodology or metrics used to measure citation accuracy.', 'grounded': True, 'pages': ['1', '4']}


## Next

Phase 7 adds formal evaluation (RAGAS/TruLens/custom citation metric) on
top of this `/ask` endpoint's output, replacing the manual "read the
answer and judge it" checks used in Phases 4–6 so far.